# Hybrid MPI + OpenMP Parallel FFT for Large Speech Signals
PDC semester project #11 — M Tashkeel Pasha (23k-2014), M Kashan (23k-0043), Hamadullah (23k-0723)

This notebook is the **Linux build + data + correctness** environment:
1. installs Open MPI, clones the repo, builds all three versions (`gcc -O3`, `-fopenmp`, `mpicc`)
2. downloads real Urdu speech (YODAS `ur000`, CC-BY-3.0) and packs it into a 2^24-sample signal (~17.5 min at 16 kHz)
3. verifies every version against `numpy.fft` on random data
4. times them on the Urdu signal and measures the MPI transpose (communication) share
5. bonus: the same FFT on the T4 GPU via cuFFT, as an industry comparison row

**Honest hardware note:** free Colab gives ~2 vCPUs, so speedup here tops out near 2x and all MPI ranks share one machine (label results "1 node"). The report's real scaling curves come from the group laptops — `bench/run_omp_scaling.ps1` (10C/16T) and 2–3 laptops over LAN with MS-MPI.

In [ ]:
# 1. Setup: MPI toolchain + repo (fresh each session — Colab VMs are wiped)
!apt -qq install -y openmpi-bin libopenmpi-dev > /dev/null
!rm -rf pdc && git clone -q https://github.com/TashkeelPasha/pdc
%cd pdc
!pip -q install soundfile huggingface_hub
!gcc --version | head -1 && mpicc --version | head -1

In [ ]:
# 2. What hardware did we get?
!echo "vCPUs: $(nproc)" && free -h | head -2
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU runtime'

In [ ]:
# 3. Real Urdu speech: two YODAS ur000 shards (~600 MB ≈ 6 h of audio, plenty for 2^24–2^26)
#    Set FULL=True for all 8 shards (2.29 GB, 22.4 h) when building 2^28 later.
FULL = False
from huggingface_hub import snapshot_download
pats = ['data/ur000/audio/*'] if FULL else ['data/ur000/audio/00000000.tar.gz', 'data/ur000/audio/00000001.tar.gz']
snapshot_download('espnet/yodas', repo_type='dataset', allow_patterns=pats, local_dir='data/yodas')
!mkdir -p data/audio && for f in data/yodas/data/ur000/audio/*.tar.gz; do tar -xzf "$f" -C data/audio; done
!find data/audio -type f | wc -l

In [ ]:
# 4. Pack the clips into one 2^24-sample complex signal (256 MiB .bin)
!python3 tools/prepare_signal.py data/audio 24 data/urdu_2e24.bin

In [ ]:
# 5. Build all three versions and verify them against numpy on random input
!make
!python3 tools/check_fft.py 16 --mpi-procs 2

In [ ]:
# 6. Time all versions on the real Urdu signal -> results/colab.csv
import subprocess, os, pathlib
pathlib.Path('results').mkdir(exist_ok=True)
P, SIG = 24, 'data/urdu_2e24.bin'
runs = [(['./build/fft_seq', str(P), SIG], {}),
        (['./build/fft_omp', str(P), SIG], {'OMP_NUM_THREADS': '1'}),
        (['./build/fft_omp', str(P), SIG], {'OMP_NUM_THREADS': '2'})]
for n in (1, 2, 4):
    runs.append((['mpiexec', '-n', str(n), '--allow-run-as-root', '--oversubscribe',
                  './build/fft_hybrid', str(P), SIG], {'OMP_NUM_THREADS': '1'}))
with open('results/colab.csv', 'a') as csv:
    for cmd, env in runs:
        for rep in range(2):
            r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, **env})
            for line in r.stdout.splitlines():
                print(line)
                if line.startswith('RESULT,'):
                    csv.write(line + '\n')
print('\nsaved -> results/colab.csv (download this; the laptops produce their own csvs)')

In [ ]:
# 7. Graphs (time / speedup / efficiency) from every csv present
!python3 bench/plot_results.py
from IPython.display import Image, display
import glob
for png in sorted(glob.glob('results/*_24.png')):
    display(Image(png))

In [ ]:
# 8. BONUS ROW — same signal through cuFFT on the T4 GPU (library on specialised
#    hardware; the course project is building + scaling the algorithm on CPUs).
import numpy as np, cupy as cp, time
x = np.fromfile('data/urdu_2e24.bin', dtype=np.complex128)
g = cp.asarray(x); cp.fft.fft(g); cp.cuda.Stream.null.synchronize()  # warm-up + plan
t0 = time.perf_counter(); y = cp.fft.fft(g); cp.cuda.Stream.null.synchronize()
gpu_ms = (time.perf_counter() - t0) * 1e3
err = float(cp.max(cp.abs(y - cp.asarray(np.fft.fft(x)))) / cp.max(cp.abs(y)))
print(f'cuFFT on T4: 2^24 points in {gpu_ms:.1f} ms  (max rel err vs numpy {err:.2e})')

## Next steps (off Colab)
- **Laptop (10C/16T):** `.\\build_all.ps1` then `.\\bench\\run_omp_scaling.ps1 -Input data\\urdu_2e24.bin` → `results/omp_laptop.csv`
- **2–3 laptops over LAN (MS-MPI):** same `fft_hybrid.exe` + same .bin on every machine, `mpiexec -hosts ...` → `results/mpi_lan.csv`
- Re-run cell 7 with all csvs in `results/` to get the final report graphs
- Scale up: `FULL=True` in cell 3, build 2^26/2^28 signals, repeat (mind RAM: the hybrid needs ~4x the signal size in total)